## Example 1: Basic ArcGIS Feature Service Access

In [ ]:
# Example 1: Basic ArcGIS Feature Service Access

import esridump
import json

# Demonstrate basic ArcGIS Feature Service data extraction
print("=== Basic ArcGIS Feature Service Access ===")

try:
    service_url = "https://earthquake.usgs.gov/arcgis/rest/services/eq/Recent_Earthquakes/MapServer/0"
    
    print("=== Service Information ===")
    print(f"Target service: USGS Recent Earthquakes")
    print(f"Service URL: {service_url}")
    
    # Create dumper instance
    print(f"\n=== Creating EsriDumper Instance ===")
    
    # Initialize the dumper with basic configuration (construction is lazy: nothing is requested yet)
    dumper = esridump.EsriDumper(service_url)
    
    print(f"✓ EsriDumper created successfully")
    
    # Layer metadata comes from get_metadata(); esridump 1.13 keeps the URL private (no `dumper.url`)
    print(f"\n=== Service Metadata ===")
    metadata = dumper.get_metadata()
    feature_total = dumper.get_feature_count()
    print(f"Dumper instance type: {type(dumper).__name__}")
    print(f"Layer name: {metadata.get('name')}")
    print(f"Available fields: {len(metadata.get('fields') or [])}")
    print(f"Geometry type: {metadata.get('geometryType')}")
    print(f"Features in layer: {feature_total}")
    assert metadata.get("geometryType") == "esriGeometryPoint", metadata.get("geometryType")
    assert feature_total > 0, "the USGS recent-earthquakes layer reported no features"
    
    # Fetch a small sample of features
    print(f"\n=== Feature Sampling ===")
    
    feature_count = 0
    sample_features = []
    
    for feature in dumper:
        sample_features.append(feature)
        feature_count += 1
        
        # Limit to first 5 features for demonstration
        if feature_count >= 5:
            break
    
    print(f"✓ Successfully fetched {len(sample_features)} sample features")
    assert len(sample_features) == 5, f"expected 5 sample features, got {len(sample_features)}"
    
    # Examine first feature structure: esridump yields GeoJSON Feature dicts
    first_feature = sample_features[0]
    
    print(f"\n=== Sample Feature Structure ===")
    print(f"Feature type: {type(first_feature)}")
    print(f"Feature keys: {list(first_feature.keys())}")
    
    geometry = first_feature['geometry']
    properties = first_feature['properties']
    print(f"Geometry type: {geometry.get('type')}")
    print(f"Properties count: {len(properties)}")
    print(f"Sample property keys: {list(properties.keys())[:5]}")
    print(f"First event: M{properties['mag']} {properties['place']}")
    assert first_feature['type'] == 'Feature', first_feature['type']
    assert {'mag', 'place'} <= set(properties), sorted(properties)
    assert all(f['geometry']['type'] == 'Point' for f in sample_features), [f['geometry'] for f in sample_features]
    
    # Show feature statistics
    print(f"\n=== Feature Statistics ===")
    print(f"Total features sampled: {len(sample_features)}")
    print(f"Features are consistent: {all(type(f) == type(sample_features[0]) for f in sample_features)}")

    print("Basic ArcGIS Feature Service access demonstration completed successfully")

except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Demonstration completed with note: {e}")
    raise AssertionError(
        f"esridump example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Feature Service Data Filtering and Querying

In [ ]:
# Example 2: Feature Service Data Filtering and Querying

import esridump
import json

# Demonstrate advanced querying and filtering of ArcGIS Feature Service data
print("=== Feature Service Data Filtering and Querying ===")

try:
    print("=== Query Parameter Configuration ===")
    
    base_service_url = "https://tigerweb.geo.census.gov/arcgis/rest/services/Generalized_ACS2023/State_County/MapServer/13"
    
    print(f"Target service: US Census TIGERweb counties (1:20M)")
    print(f"Base URL: {base_service_url}")
    
    total_counties = esridump.EsriDumper(base_service_url).get_feature_count()
    print(f"Counties in layer: {total_counties}")
    assert total_counties > 3000, f"the county layer reported only {total_counties} features"
    
    # Demonstrate different query configurations; `check` is what every returned feature must satisfy
    query_examples = [
        {
            "name": "Basic Query",
            "params": {},
            "description": "Default query with no filters"
        },
        {
            "name": "Attribute Filter",
            "params": {"where": "STATE = '06'"},
            "description": "Counties in California (state FIPS 06)",
            "expected_matches": 58,
            "check": lambda props: props["STATE"] == "06"
        },
        {
            "name": "Field Selection",
            "params": {"fields": "NAME,STATE,AREALAND"},
            "description": "Select specific fields only",
            "check": lambda props: set(props) == {"NAME", "STATE", "AREALAND"}
        },
        {
            "name": "Geometry Bounds",
            "params": {"bbox": "-125,32,-114,42"},  # California approximate bounds
            "description": "Spatial filter for California region"
        },
        {
            "name": "Combined Filters",
            "params": {
                "where": "AREALAND > 10000000000",
                "limit": 10
            },
            "description": "Land area over 10,000 km² with a 10-record page size",
            "check": lambda props: props["AREALAND"] > 10_000_000_000
        }
    ]
    
    print(f"\n=== Query Examples ===")
    
    for i, query_config in enumerate(query_examples, 1):
        print(f"\n{i}. {query_config['name']}")
        print(f"   Description: {query_config['description']}")
        print(f"   Parameters: {query_config['params']}")
        
        # Translate the parameters into esridump arguments: server-side filters go in
        # `extra_query_args`, the field list in `fields`, the page size in `max_page_size`
        extra_query_args = {}
        dumper_kwargs = {}
        for key, value in query_config['params'].items():
            if key == "bbox":
                # Envelope in WGS84 lon/lat; inSR is required because the layer is Web Mercator
                extra_query_args.update({
                    "geometry": value,
                    "geometryType": "esriGeometryEnvelope",
                    "spatialRel": "esriSpatialRelIntersects",
                    "inSR": "4326",
                })
            elif key == "where":
                extra_query_args["where"] = value
            elif key == "fields":
                dumper_kwargs["fields"] = value.split(",")
                # esridump drops `fields` on its offset-pagination path (openaddresses/pyesridump#50); object-ID pagination honours them
                dumper_kwargs["paginate_oid"] = True
            elif key == "limit":
                dumper_kwargs["max_page_size"] = value
        
        dumper = esridump.EsriDumper(base_service_url, extra_query_args=extra_query_args, **dumper_kwargs)
        matching = dumper.get_feature_count()
        print(f"   Matching features: {matching}")
        
        # Sample features from this configuration
        sample_count = 0
        sample_features = []
        
        for feature in dumper:
            sample_features.append(feature)
            sample_count += 1
            
            # Limit sample size
            if sample_count >= 3:
                break
        
        print(f"   ✓ Retrieved {len(sample_features)} sample features")
        assert len(sample_features) == 3, (
            f"{query_config['name']}: expected 3 sample features, got {len(sample_features)}")
        
        # Show feature properties
        properties = sample_features[0]['properties']
        print(f"   Sample properties: {list(properties.keys())[:5]}")
        print(f"   Example county: {properties['NAME']}")
        print(f"   Land area: {properties['AREALAND']:,} m²")
        
        # The server must have applied what was asked for
        assert all(f['geometry']['type'] in ("Polygon", "MultiPolygon") for f in sample_features)
        if extra_query_args:
            assert 0 < matching < total_counties, (
                f"{query_config['name']}: the filter matched {matching} of {total_counties}")
        else:
            assert matching == total_counties, (matching, total_counties)
        if "expected_matches" in query_config:
            assert matching == query_config["expected_matches"], (query_config['name'], matching)
        if "check" in query_config:
            assert all(query_config["check"](f['properties']) for f in sample_features), (
                query_config['name'], [f['properties'] for f in sample_features])
    
    # Demonstrate programmatic query building
    print(f"\n=== Programmatic Query Building ===")
    
    class QueryBuilder:
        """Helper class for building ArcGIS REST API queries."""
        
        def __init__(self, base_url):
            self.base_url = base_url
            self.params = {}
        
        def where(self, condition):
            """Add WHERE clause for attribute filtering."""
            self.params['where'] = condition
            return self
        
        def fields(self, field_list):
            """Select specific output fields."""
            if isinstance(field_list, list):
                field_list = ','.join(field_list)
            self.params['outFields'] = field_list
            return self
        
        def bbox(self, xmin, ymin, xmax, ymax):
            """Add bounding box spatial filter (WGS84 lon/lat)."""
            bbox_str = f"{xmin},{ymin},{xmax},{ymax}"
            self.params['geometry'] = bbox_str
            self.params['geometryType'] = 'esriGeometryEnvelope'
            self.params['spatialRel'] = 'esriSpatialRelIntersects'
            self.params['inSR'] = '4326'
            return self
        
        def limit(self, count):
            """Limit number of returned features."""
            self.params['resultRecordCount'] = count
            return self
        
        def build_dumper(self):
            """Create EsriDumper instance with configured parameters."""
            extra_query_args = {k: v for k, v in self.params.items() if k not in ('outFields', 'resultRecordCount')}
            kwargs = {}
            if 'outFields' in self.params:
                kwargs['fields'] = self.params['outFields'].split(',')
                kwargs['paginate_oid'] = True  # see the paginate_oid note in the query loop
            if 'resultRecordCount' in self.params:
                kwargs['max_page_size'] = self.params['resultRecordCount']
            return esridump.EsriDumper(self.base_url, extra_query_args=extra_query_args, **kwargs)
        
        def get_query_string(self):
            """Get query parameters as string."""
            return "&".join([f"{k}={v}" for k, v in self.params.items()])
    
    # Example of using QueryBuilder
    print(f"QueryBuilder usage examples:")
    
    builder = QueryBuilder(base_service_url)
    
    # Build an attribute query
    query1 = (builder
             .where("AREALAND > 10000000000")
             .limit(5))
    
    print(f"  Attribute query: {query1.get_query_string()}")
    
    # Build a spatial query
    builder2 = QueryBuilder(base_service_url)
    query2 = (builder2
             .bbox(-125, 32, -114, 42)  # California bounds
             .fields(['NAME', 'AREALAND']))
    
    print(f"  Spatial query: {query2.get_query_string()}")
    
    # Test query execution: the first feature of each query must satisfy it
    first1 = next(iter(query1.build_dumper()), None)
    first2 = next(iter(query2.build_dumper()), None)
    assert first1 is not None and first2 is not None, "a QueryBuilder query returned no features"
    print(f"  ✓ Attribute query returned {first1['properties']['NAME']} ({first1['properties']['AREALAND']:,} m²)")
    print(f"  ✓ Spatial query returned {first2['properties']}")
    assert first1['properties']['AREALAND'] > 10_000_000_000, first1['properties']
    assert set(first2['properties']) == {'NAME', 'AREALAND'}, first2['properties']
    
    # Query optimization tips
    print(f"\n=== Query Optimization Tips ===")
    
    optimization_tips = [
        "Use specific field selection to reduce data transfer",
        "Apply WHERE clauses to filter on the server side",
        "Use spatial filters (bbox) to limit geographic scope",
        "Set appropriate result limits for large datasets",
        "Consider indexed fields for faster WHERE queries",
        "Use appropriate spatial reference systems",
        "Batch requests for large data extractions"
    ]
    
    for i, tip in enumerate(optimization_tips, 1):
        print(f"  {i}. {tip}")

    print("Feature Service data filtering and querying demonstration completed successfully")

except Exception as e:
    print(f"Querying demonstration completed with note: {e}")
    raise AssertionError(
        f"esridump example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Batch Data Extraction and Processing

In [ ]:
# Example 3: Batch Data Extraction and Processing

import esridump
import json
import time

# Demonstrate batch data extraction and processing workflows
print("=== Batch Data Extraction and Processing ===")

try:
    print("=== Batch Processing Configuration ===")
    
    service_url = "https://tigerweb.geo.census.gov/arcgis/rest/services/Generalized_ACS2023/State_County/MapServer/9"
    
    print(f"Target service: US Census TIGERweb states (1:20M)")
    print(f"Service URL: {service_url}")
    
    # Batch processing parameters
    batch_config = {
        "batch_size": 10,        # Features per batch
        "max_features": 50,      # Total feature limit for demo
        "process_delay": 0.1,    # Delay between batches (seconds)
        "retry_attempts": 3,     # Retry failed requests
        "timeout": 30            # Request timeout (seconds)
    }
    
    print(f"Batch configuration: {batch_config}")
    
    # Create batch processor
    print(f"\n=== Batch Processor Implementation ===")
    
    class BatchProcessor:
        """Batch processor for ArcGIS Feature Service data."""
        
        def __init__(self, service_url, config=None):
            self.service_url = service_url
            self.config = config or {}
            self.stats = {
                "total_features": 0,
                "batches_processed": 0,
                "processing_time": 0,
                "errors": 0
            }
        
        def process_batch(self, features):
            """Process a batch of features."""
            processed = []
            
            for feature in features:
                try:
                    # Example processing: extract key information
                    processed_feature = self._process_single_feature(feature)
                    processed.append(processed_feature)
                    
                except Exception as e:
                    self.stats["errors"] += 1
                    print(f"    Feature processing error: {e}")
            
            return processed
        
        def _process_single_feature(self, feature):
            """Process individual feature."""
            if isinstance(feature, dict):
                result = {
                    "feature_type": type(feature).__name__,
                    "has_geometry": "geometry" in feature,
                    "has_properties": "properties" in feature,
                    "timestamp": time.time()
                }
                
                # Extract geometry info
                if "geometry" in feature and feature["geometry"]:
                    geometry = feature["geometry"]
                    result["geometry_type"] = geometry.get("type", "Unknown")
                
                # Extract property info
                if "properties" in feature and feature["properties"]:
                    properties = feature["properties"]
                    result["property_count"] = len(properties)
                    result["property_keys"] = list(properties.keys())[:5]  # First 5 keys
                    
                    # Extract specific fields if available
                    for field in ["NAME", "STUSAB", "STATE", "AREALAND"]:
                        if field in properties:
                            result[field.lower()] = properties[field]
                
                return result
            
            return {"raw_feature": str(feature)[:100]}  # Truncated string representation
        
        def extract_all(self):
            """Extract and process all features in batches."""
            start_time = time.time()
            
            try:
                dumper = esridump.EsriDumper(self.service_url)
                
                current_batch = []
                processed_features = []
                
                for feature in dumper:
                    current_batch.append(feature)
                    self.stats["total_features"] += 1
                    
                    # Process batch when it reaches the configured size
                    if len(current_batch) >= self.config.get("batch_size", 10):
                        batch_result = self.process_batch(current_batch)
                        processed_features.extend(batch_result)
                        
                        self.stats["batches_processed"] += 1
                        print(f"    Processed batch {self.stats['batches_processed']} ({len(current_batch)} features)")
                        
                        # Clear batch
                        current_batch = []
                        
                        # Add processing delay
                        if self.config.get("process_delay", 0) > 0:
                            time.sleep(self.config["process_delay"])
                    
                    # Stop if max features reached
                    if self.stats["total_features"] >= self.config.get("max_features", float('inf')):
                        break
                
                # Process remaining features in final batch
                if current_batch:
                    batch_result = self.process_batch(current_batch)
                    processed_features.extend(batch_result)
                    self.stats["batches_processed"] += 1
                    print(f"    Processed final batch {self.stats['batches_processed']} ({len(current_batch)} features)")
                
                self.stats["processing_time"] = time.time() - start_time
                return processed_features
                
            except Exception as e:
                self.stats["processing_time"] = time.time() - start_time
                print(f"Batch extraction error: {e}")
                raise
        
        def get_stats(self):
            """Get processing statistics."""
            stats = self.stats.copy()
            
            if stats["processing_time"] > 0:
                stats["features_per_second"] = stats["total_features"] / stats["processing_time"]
                stats["batches_per_second"] = stats["batches_processed"] / stats["processing_time"]
            
            return stats
    
    # Execute batch processing
    print(f"\n=== Batch Processing Execution ===")
    
    processor = BatchProcessor(service_url, batch_config)
    
    print(f"Starting batch extraction...")
    processed_data = processor.extract_all()
    
    print(f"\n✓ Batch processing completed")
    
    # Show processing statistics
    stats = processor.get_stats()
    
    print(f"\n=== Processing Statistics ===")
    print(f"Total features: {stats['total_features']}")
    print(f"Batches processed: {stats['batches_processed']}")
    print(f"Processing time: {stats['processing_time']:.2f} seconds")
    print(f"Errors: {stats['errors']}")
    
    if stats.get('features_per_second'):
        print(f"Features per second: {stats['features_per_second']:.2f}")
    if stats.get('batches_per_second'):
        print(f"Batches per second: {stats['batches_per_second']:.2f}")
    
    # The layer holds 52 states, so the 50-feature cap must be reached in 5 full batches of 10
    assert stats['total_features'] == batch_config['max_features'], stats
    assert stats['batches_processed'] == batch_config['max_features'] // batch_config['batch_size'], stats
    assert stats['errors'] == 0, stats
    assert len(processed_data) == batch_config['max_features'], len(processed_data)
    assert all(f.get('geometry_type') in ('Polygon', 'MultiPolygon') and f.get('name') for f in processed_data), (
        processed_data[:3])
    
    # Show sample of processed data
    print(f"\n=== Sample Processed Data ===")
    print(f"Processed features: {len(processed_data)}")
    
    # Show first few processed features
    for i, feature in enumerate(processed_data[:3]):
        print(f"\nFeature {i+1}:")
        for key, value in feature.items():
            if isinstance(value, list) and len(value) > 5:
                print(f"  {key}: {value[:5]}... ({len(value)} items)")
            else:
                print(f"  {key}: {value}")
    
    # Data export and persistence patterns
    print(f"\n=== Data Export Patterns ===")
    
    export_patterns = {
        "JSON Export": {
            "description": "Export processed data as JSON",
            "example": "json.dump(processed_data, open('output.json', 'w'))"
        },
        "CSV Export": {
            "description": "Convert to tabular format for CSV",
            "example": "pd.DataFrame(processed_data).to_csv('output.csv')"
        },
        "GeoJSON Export": {
            "description": "Maintain spatial format",
            "example": "Export features with geometry preserved"
        },
        "Database Insert": {
            "description": "Bulk insert into database",
            "example": "INSERT INTO table VALUES ... (batch mode)"
        },
        "Streaming Export": {
            "description": "Process and export on-the-fly",
            "example": "Write features as they are processed"
        }
    }
    
    for pattern_name, pattern_info in export_patterns.items():
        print(f"\n{pattern_name}:")
        print(f"  Description: {pattern_info['description']}")
        print(f"  Example: {pattern_info['example']}")
    
    # Performance optimization tips
    print(f"\n=== Performance Optimization Tips ===")
    
    optimization_tips = [
        "Use appropriate batch sizes (10-100 features typically optimal)",
        "Implement retry logic for network failures",
        "Add processing delays to respect server limits",
        "Use connection pooling for multiple requests",
        "Process features incrementally to reduce memory usage",
        "Implement progress monitoring for long-running extractions",
        "Cache frequently accessed service metadata",
        "Use concurrent processing for independent batches"
    ]
    
    for i, tip in enumerate(optimization_tips, 1):
        print(f"  {i}. {tip}")

    print("Batch data extraction and processing demonstration completed successfully")

except Exception as e:
    print(f"Batch processing demonstration completed with note: {e}")
    raise AssertionError(
        f"esridump example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Error Handling and Resilient Data Access

In [ ]:
# Example 4: Error Handling and Resilient Data Access

import esridump
import time
import json
from typing import List, Dict, Optional

# Demonstrate robust error handling and resilient data access patterns
print("=== Error Handling and Resilient Data Access ===")

try:
    print("=== Error Handling Strategies ===")
    
    class ResilientEsriClient:
        """Resilient client for ArcGIS Feature Service access with comprehensive error handling."""
        
        def __init__(self, service_url: str, config: Optional[Dict] = None):
            self.service_url = service_url
            self.config = {**self._default_config(), **(config or {})}
            self.error_log = []
            self.stats = {
                "total_attempts": 0,
                "successful_requests": 0,
                "failed_requests": 0,
                "retries": 0,
                "total_features": 0
            }
        
        def _default_config(self) -> Dict:
            return {
                "max_retries": 3,
                "retry_delay": 1.0,
                "backoff_multiplier": 2.0,
                "request_timeout": 30,
                "max_errors": 10,
                "validate_responses": True
            }
        
        def _log_error(self, error_type: str, message: str, context: Optional[Dict] = None):
            """Log error with context information."""
            error_entry = {
                "timestamp": time.time(),
                "error_type": error_type,
                "message": message,
                "context": context or {}
            }
            self.error_log.append(error_entry)
            
            # Limit error log size
            if len(self.error_log) > 100:
                self.error_log = self.error_log[-50:]  # Keep last 50 errors
        
        def _validate_service_url(self, url: str) -> bool:
            """Basic service URL validation."""
            try:
                if not url:
                    return False
                
                # Check for basic URL structure
                if not url.startswith(('http://', 'https://')):
                    return False
                
                # Check for ArcGIS service patterns
                valid_patterns = ['/MapServer/', '/FeatureServer/', '/rest/services/']
                if not any(pattern in url for pattern in valid_patterns):
                    return False
                
                return True
                
            except Exception as e:
                self._log_error("URL_VALIDATION", f"URL validation error: {e}")
                return False
        
        def _create_dumper_with_retry(self) -> Optional[esridump.EsriDumper]:
            """Create EsriDumper with retry logic."""
            
            if not self._validate_service_url(self.service_url):
                self._log_error("INVALID_URL", f"Invalid service URL: {self.service_url}")
                return None
            
            for attempt in range(self.config["max_retries"] + 1):
                try:
                    self.stats["total_attempts"] += 1
                    
                    # Attempt to create dumper
                    dumper = esridump.EsriDumper(self.service_url)
                    
                    self.stats["successful_requests"] += 1
                    return dumper
                    
                except Exception as e:
                    self.stats["failed_requests"] += 1
                    
                    if attempt < self.config["max_retries"]:
                        self.stats["retries"] += 1
                        
                        # Calculate retry delay with backoff
                        delay = self.config["retry_delay"] * (self.config["backoff_multiplier"] ** attempt)
                        
                        self._log_error(
                            "DUMPER_CREATION_RETRY",
                            f"Attempt {attempt + 1} failed, retrying in {delay:.1f}s: {e}",
                            {"attempt": attempt + 1, "delay": delay}
                        )
                        
                        time.sleep(delay)
                    else:
                        self._log_error(
                            "DUMPER_CREATION_FAILED",
                            f"All attempts failed: {e}",
                            {"total_attempts": attempt + 1}
                        )
            
            return None
        
        def _validate_feature(self, feature: Dict) -> bool:
            """Validate feature structure."""
            if not self.config["validate_responses"]:
                return True
            
            try:
                # Basic feature validation
                if not isinstance(feature, dict):
                    return False
                
                # Check for expected structure
                expected_keys = ["type", "geometry", "properties"]
                if not any(key in feature for key in expected_keys):
                    return False
                
                return True
                
            except Exception as e:
                self._log_error("FEATURE_VALIDATION", f"Feature validation error: {e}")
                return False
        
        def extract_features_safely(self, max_features: Optional[int] = None) -> List[Dict]:
            """Extract features with comprehensive error handling."""
            
            print(f"Starting resilient feature extraction...")
            print(f"Service URL: {self.service_url}")
            print(f"Configuration: {self.config}")
            
            # Create dumper with retry logic
            dumper = self._create_dumper_with_retry()
            
            if not dumper:
                print("✗ Failed to create dumper after all retry attempts")
                return []
            
            print(f"✓ Successfully created dumper")
            
            features = []
            error_count = 0
            
            try:
                for i, feature in enumerate(dumper):
                    try:
                        # Validate feature
                        if not self._validate_feature(feature):
                            error_count += 1
                            self._log_error(
                                "INVALID_FEATURE",
                                f"Feature {i} failed validation",
                                {"feature_index": i}
                            )
                            continue
                        
                        features.append(feature)
                        self.stats["total_features"] += 1
                        
                        # Stop if max features reached
                        if max_features and len(features) >= max_features:
                            print(f"Reached maximum feature limit: {max_features}")
                            break
                        
                        # Stop if too many errors
                        if error_count >= self.config["max_errors"]:
                            self._log_error(
                                "MAX_ERRORS_REACHED",
                                f"Stopped extraction due to {error_count} errors"
                            )
                            break
                        
                        # Progress reporting
                        if (i + 1) % 10 == 0:
                            print(f"  Processed {i + 1} features, {len(features)} valid")
                        
                    except Exception as e:
                        error_count += 1
                        self._log_error(
                            "FEATURE_PROCESSING",
                            f"Error processing feature {i}: {e}",
                            {"feature_index": i}
                        )
                        
                        if error_count >= self.config["max_errors"]:
                            break
            
            except Exception as e:
                self._log_error("EXTRACTION_FAILED", f"Feature extraction failed: {e}")
            
            print(f"✓ Extraction completed: {len(features)} features extracted")
            print(f"Errors encountered: {error_count}")
            
            return features
        
        def get_error_summary(self) -> Dict:
            """Get comprehensive error summary."""
            if not self.error_log:
                return {"message": "No errors logged"}
            
            # Group errors by type
            error_types = {}
            for error in self.error_log:
                error_type = error["error_type"]
                error_types[error_type] = error_types.get(error_type, 0) + 1
            
            # Recent errors (last 5)
            recent_errors = self.error_log[-5:] if len(self.error_log) >= 5 else self.error_log
            
            return {
                "total_errors": len(self.error_log),
                "error_types": error_types,
                "recent_errors": [
                    {
                        "type": error["error_type"],
                        "message": error["message"][:100],  # Truncate long messages
                        "timestamp": error["timestamp"]
                    }
                    for error in recent_errors
                ],
                "stats": self.stats
            }
    
    # Test resilient client with various scenarios
    print(f"\n=== Resilient Client Testing ===")
    
    # Test with valid service
    print(f"\n1. Testing with valid service:")
    
    valid_service = "https://tigerweb.geo.census.gov/arcgis/rest/services/Generalized_ACS2023/State_County/MapServer/9"
    
    client = ResilientEsriClient(valid_service, {
        "max_retries": 2,
        "retry_delay": 0.5,
        "max_errors": 5
    })
    
    features = client.extract_features_safely(max_features=10)
    
    print(f"  ✓ Extracted {len(features)} features successfully")
    
    # Show error summary
    error_summary = client.get_error_summary()
    print(f"  Errors logged: {error_summary.get('total_errors', 0)}")
    
    # A live layer must yield real features, and the resilient client must have logged nothing
    assert len(features) == 10, f"expected 10 features from the live layer, got {len(features)}: {error_summary}"
    assert error_summary.get('total_errors', 0) == 0, error_summary
    assert all(f['geometry']['type'] in ('Polygon', 'MultiPolygon') for f in features), [f['geometry']['type'] for f in features]
    assert all(f['properties'].get('NAME') for f in features), [f['properties'] for f in features]
    
    # Test with invalid service URL
    print(f"\n2. Testing with invalid service URL:")
    
    invalid_service = "https://invalid-service-url/nonexistent"
    
    client = ResilientEsriClient(invalid_service, {
        "max_retries": 1,
        "retry_delay": 0.1
    })
    
    features = client.extract_features_safely(max_features=5)
    
    print(f"  Features extracted: {len(features)}")
    
    error_summary = client.get_error_summary()
    print(f"  Errors logged: {error_summary.get('total_errors', 0)}")
    
    if error_summary.get('error_types'):
        print(f"  Error types: {list(error_summary['error_types'].keys())}")
    
    # URL validation must reject it before any request, and say why
    assert features == [], features
    assert error_summary.get('error_types') == {'INVALID_URL': 1}, error_summary
    
    # Error handling best practices
    print(f"\n=== Error Handling Best Practices ===")
    
    best_practices = [
        "Implement exponential backoff for retries",
        "Validate service URLs before attempting connections",
        "Set reasonable timeouts for network requests",
        "Log errors with context for debugging",
        "Implement circuit breaker patterns for repeated failures",
        "Validate response data structure and content",
        "Provide graceful degradation when services are unavailable",
        "Monitor error rates and patterns in production",
        "Use connection pooling for multiple requests",
        "Implement rate limiting to respect server limits"
    ]
    
    for i, practice in enumerate(best_practices, 1):
        print(f"  {i}. {practice}")
    
    # Common error scenarios and solutions
    print(f"\n=== Common Error Scenarios ===")
    
    error_scenarios = {
        "Network Timeout": {
            "description": "Request takes too long to complete",
            "solution": "Increase timeout, implement retries"
        },
        "Service Unavailable": {
            "description": "ArcGIS service is down or overloaded",
            "solution": "Retry with backoff, use alternative endpoints"
        },
        "Invalid Credentials": {
            "description": "Authentication failure for secured services",
            "solution": "Refresh tokens, validate credentials"
        },
        "Rate Limiting": {
            "description": "Too many requests in short time",
            "solution": "Implement delays, use service limits"
        },
        "Malformed Response": {
            "description": "Unexpected or corrupted data format",
            "solution": "Validate responses, handle parsing errors"
        },
        "Memory Issues": {
            "description": "Large datasets exceed available memory",
            "solution": "Stream processing, pagination"
        }
    }
    
    for scenario, info in error_scenarios.items():
        print(f"\n{scenario}:")
        print(f"  Problem: {info['description']}")
        print(f"  Solution: {info['solution']}")

    print("Error handling and resilient data access demonstration completed successfully")

except Exception as e:
    print(f"Error handling demonstration completed with note: {e}")
    raise AssertionError(
        f"esridump example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Production Data Integration Workflow

In [ ]:
# Example 5: Production Data Integration Workflow

import esridump
import json
import time
from typing import Dict, List, Optional, Any
from dataclasses import dataclass
from datetime import datetime, timedelta

# Demonstrate production-ready data integration workflow
print("=== Production Data Integration Workflow ===")

try:
    @dataclass
    class DataSourceConfig:
        """Configuration for ArcGIS data source."""
        name: str
        service_url: str
        update_frequency: str  # 'hourly', 'daily', 'weekly'
        fields_of_interest: List[str]
        spatial_filter: Optional[Dict[str, Any]] = None
        max_features: Optional[int] = None
        enabled: bool = True
    
    @dataclass
    class IntegrationJob:
        """Data integration job tracking."""
        job_id: str
        source_name: str
        start_time: float
        end_time: Optional[float] = None
        features_processed: int = 0
        status: str = "RUNNING"  # RUNNING, COMPLETED, FAILED
        errors: List[str] = None
        
        def __post_init__(self):
            if self.errors is None:
                self.errors = []
    
    class ProductionDataIntegrator:
        """Production-ready ArcGIS data integration system."""
        
        def __init__(self, config: Dict[str, Any]):
            self.config = config
            self.data_sources = {}
            self.job_history = []
            self.cache = {}
            self.stats = {
                "total_jobs": 0,
                "successful_jobs": 0,
                "failed_jobs": 0,
                "total_features_processed": 0,
                "uptime_start": time.time()
            }
        
        def register_data_source(self, source_config: DataSourceConfig):
            """Register a new data source."""
            self.data_sources[source_config.name] = source_config
            print(f"✓ Registered data source: {source_config.name}")
        
        def _should_update_source(self, source_name: str) -> bool:
            """Check if data source needs updating based on frequency."""
            if source_name not in self.cache:
                return True
            
            source_config = self.data_sources[source_name]
            last_update = self.cache[source_name].get('last_update', 0)
            current_time = time.time()
            
            # Define update intervals
            intervals = {
                'hourly': 3600,
                'daily': 86400,
                'weekly': 604800
            }
            
            interval = intervals.get(source_config.update_frequency, 3600)
            
            return (current_time - last_update) > interval
        
        def _extract_features(self, source_config: DataSourceConfig) -> List[Dict]:
            """Extract features from ArcGIS service."""
            try:
                # Create dumper
                dumper = esridump.EsriDumper(source_config.service_url)
                
                features = []
                feature_count = 0
                
                for feature in dumper:
                    # Apply field filtering if specified
                    if source_config.fields_of_interest:
                        filtered_feature = self._filter_feature_fields(
                            feature, source_config.fields_of_interest
                        )
                    else:
                        filtered_feature = feature
                    
                    features.append(filtered_feature)
                    feature_count += 1
                    
                    # Respect max features limit
                    if (source_config.max_features and 
                        feature_count >= source_config.max_features):
                        break
                
                return features
                
            except Exception as e:
                raise Exception(f"Feature extraction failed: {e}")
        
        def _filter_feature_fields(self, feature: Dict, fields_of_interest: List[str]) -> Dict:
            """Filter feature to include only specified fields."""
            if not isinstance(feature, dict) or 'properties' not in feature:
                return feature
            
            filtered_feature = feature.copy()
            
            if 'properties' in feature and feature['properties']:
                filtered_properties = {}
                
                for field in fields_of_interest:
                    if field in feature['properties']:
                        filtered_properties[field] = feature['properties'][field]
                
                filtered_feature['properties'] = filtered_properties
            
            return filtered_feature
        
        def _validate_and_transform_data(self, features: List[Dict], source_name: str) -> List[Dict]:
            """Validate and transform extracted data."""
            transformed_features = []
            
            for i, feature in enumerate(features):
                try:
                    # Basic validation
                    if not isinstance(feature, dict):
                        continue
                    
                    # Add metadata
                    transformed_feature = feature.copy()
                    
                    if 'properties' not in transformed_feature:
                        transformed_feature['properties'] = {}
                    
                    # Add integration metadata
                    transformed_feature['properties']['_source'] = source_name
                    transformed_feature['properties']['_extracted_at'] = datetime.now().isoformat()
                    transformed_feature['properties']['_feature_index'] = i
                    
                    # Data quality checks
                    quality_score = self._calculate_quality_score(transformed_feature)
                    transformed_feature['properties']['_quality_score'] = quality_score
                    
                    # Only include high-quality features
                    if quality_score >= 0.7:
                        transformed_features.append(transformed_feature)
                    
                except Exception as e:
                    print(f"    Warning: Feature {i} transformation failed: {e}")
                    continue
            
            return transformed_features
        
        def _calculate_quality_score(self, feature: Dict) -> float:
            """Calculate data quality score for feature."""
            score = 1.0
            
            # Check for required components
            if 'geometry' not in feature or not feature['geometry']:
                score -= 0.3
            
            if 'properties' not in feature or not feature['properties']:
                score -= 0.3
            
            # Check for null/empty values in properties
            if 'properties' in feature and feature['properties']:
                properties = feature['properties']
                total_props = len(properties)
                null_props = sum(1 for v in properties.values() if v is None or v == '')
                
                if total_props > 0:
                    null_ratio = null_props / total_props
                    score -= (null_ratio * 0.4)
            
            return max(0.0, score)
        
        def _save_to_cache(self, source_name: str, features: List[Dict]):
            """Save processed features to cache."""
            self.cache[source_name] = {
                'features': features,
                'last_update': time.time(),
                'feature_count': len(features)
            }
        
        def process_data_source(self, source_name: str) -> IntegrationJob:
            """Process a single data source."""
            if source_name not in self.data_sources:
                raise ValueError(f"Unknown data source: {source_name}")
            
            source_config = self.data_sources[source_name]
            
            if not source_config.enabled:
                raise ValueError(f"Data source {source_name} is disabled")
            
            # Create job tracking
            job = IntegrationJob(
                job_id=f"{source_name}_{int(time.time())}",
                source_name=source_name,
                start_time=time.time()
            )
            
            self.job_history.append(job)
            self.stats["total_jobs"] += 1
            
            try:
                print(f"Processing data source: {source_name}")
                
                # Check if update is needed
                if not self._should_update_source(source_name):
                    print(f"  Source {source_name} is up to date, skipping...")
                    job.status = "SKIPPED"
                    job.end_time = time.time()
                    return job
                
                # Extract features
                print(f"  Extracting features...")
                raw_features = self._extract_features(source_config)
                print(f"  ✓ Extracted {len(raw_features)} raw features")
                
                # Validate and transform
                print(f"  Validating and transforming data...")
                processed_features = self._validate_and_transform_data(raw_features, source_name)
                print(f"  ✓ Processed {len(processed_features)} valid features")
                
                # Save to cache
                self._save_to_cache(source_name, processed_features)
                print(f"  ✓ Saved to cache")
                
                # Update job status
                job.features_processed = len(processed_features)
                job.status = "COMPLETED"
                job.end_time = time.time()
                
                self.stats["successful_jobs"] += 1
                self.stats["total_features_processed"] += len(processed_features)
                
                return job
                
            except Exception as e:
                job.status = "FAILED"
                job.end_time = time.time()
                job.errors.append(str(e))
                
                self.stats["failed_jobs"] += 1
                
                print(f"  ✗ Job failed: {e}")
                raise  # the job is recorded FAILED above; the example must stop on the real error, not on an assert far below
        
        def process_all_sources(self) -> Dict[str, IntegrationJob]:
            """Process all registered data sources."""
            jobs = {}
            
            print(f"Processing {len(self.data_sources)} data sources...")
            
            for source_name in self.data_sources:
                job = self.process_data_source(source_name)
                jobs[source_name] = job
            
            return jobs
        
        def get_cached_data(self, source_name: str) -> Optional[List[Dict]]:
            """Retrieve cached data for a source."""
            if source_name in self.cache:
                return self.cache[source_name]['features']
            return None
        
        def get_integration_summary(self) -> Dict:
            """Get comprehensive integration summary."""
            current_time = time.time()
            uptime = current_time - self.stats["uptime_start"]
            
            # Recent jobs (last 10)
            recent_jobs = self.job_history[-10:] if len(self.job_history) > 10 else self.job_history
            
            return {
                "system_stats": {
                    **self.stats,
                    "uptime_seconds": uptime,
                    "uptime_hours": uptime / 3600,
                    "success_rate": (
                        self.stats["successful_jobs"] / self.stats["total_jobs"] 
                        if self.stats["total_jobs"] > 0 else 0
                    )
                },
                "data_sources": {
                    "registered": len(self.data_sources),
                    "enabled": sum(1 for source in self.data_sources.values() if source.enabled),
                    "cached": len(self.cache)
                },
                "recent_jobs": [
                    {
                        "job_id": job.job_id,
                        "source": job.source_name,
                        "status": job.status,
                        "features": job.features_processed,
                        "duration": job.end_time - job.start_time if job.end_time else None
                    }
                    for job in recent_jobs
                ],
                "cache_status": {
                    source_name: {
                        "feature_count": cache_info["feature_count"],
                        "last_update": datetime.fromtimestamp(cache_info["last_update"]).isoformat(),
                        "age_hours": (current_time - cache_info["last_update"]) / 3600
                    }
                    for source_name, cache_info in self.cache.items()
                }
            }
    
    # Demo production integration workflow
    print(f"\n=== Production Integration Demo ===")
    
    # Initialize integrator
    integrator_config = {
        "max_concurrent_jobs": 3,
        "default_timeout": 300,
        "quality_threshold": 0.7
    }
    
    integrator = ProductionDataIntegrator(integrator_config)
    
    # Register data sources
    print(f"\n1. Registering data sources:")
    
    data_sources = [
        DataSourceConfig(
            name="usa_states",
            service_url="https://tigerweb.geo.census.gov/arcgis/rest/services/Generalized_ACS2023/State_County/MapServer/9",
            update_frequency="daily",
            fields_of_interest=["NAME", "STUSAB", "AREALAND"],
            max_features=10
        ),
        DataSourceConfig(
            name="usa_counties",
            service_url="https://tigerweb.geo.census.gov/arcgis/rest/services/Generalized_ACS2023/State_County/MapServer/13",
            update_frequency="weekly",
            fields_of_interest=["NAME", "STATE", "AREALAND"],
            max_features=15
        )
    ]
    
    for source in data_sources:
        integrator.register_data_source(source)
    
    # Process all sources
    print(f"\n2. Processing all data sources:")
    
    jobs = integrator.process_all_sources()
    
    # Show results
    print(f"\n3. Integration results:")
    
    for source_name, job in jobs.items():
        print(f"\n{source_name}:")
        print(f"  Job ID: {job.job_id}")
        print(f"  Status: {job.status}")
        print(f"  Features processed: {job.features_processed}")
        
        if job.end_time:
            duration = job.end_time - job.start_time
            print(f"  Duration: {duration:.2f} seconds")
        
        if job.errors:
            print(f"  Errors: {len(job.errors)}")
    
    # Show integration summary
    print(f"\n4. Integration summary:")
    
    summary = integrator.get_integration_summary()
    
    print(f"\nSystem Statistics:")
    for key, value in summary["system_stats"].items():
        if isinstance(value, float):
            print(f"  {key}: {value:.2f}")
        else:
            print(f"  {key}: {value}")
    
    print(f"\nCache Status:")
    for source, cache_info in summary["cache_status"].items():
        print(f"  {source}:")
        print(f"    Features: {cache_info['feature_count']}")
        print(f"    Age: {cache_info['age_hours']:.1f} hours")

    # Both live sources must have delivered their full quota of real, field-filtered features
    for source in data_sources:
        job = jobs.get(source.name)
        assert job is not None and job.status == "COMPLETED", (source.name, job)
        assert job.errors == [], (source.name, job.errors)
        assert job.features_processed == source.max_features, (source.name, job.features_processed)
        cached = integrator.get_cached_data(source.name)
        assert len(cached) == source.max_features, (source.name, len(cached))
        for feature in cached:
            assert feature['geometry']['type'] in ('Polygon', 'MultiPolygon'), feature['geometry']['type']
            assert set(source.fields_of_interest) <= set(feature['properties']), sorted(feature['properties'])
    assert summary["system_stats"]["success_rate"] == 1.0, summary["system_stats"]

    print("Production data integration workflow demonstration completed successfully")

except Exception as e:
    print(f"Production integration demonstration completed with note: {e}")
    raise AssertionError(
        f"esridump example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e